# 👾 PixelCNN using Tensorflow distributions

In this notebook, we'll walk through the steps required to train your own PixelCNN on the fashion MNIST dataset using Tensorflow distributions

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras only loads the dataset; this keeps TensorFlow out of the PyTorch process
from keras import datasets
import lightning as L
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

# PyTorch has no PixelCNN distribution like TensorFlow Probability, so we use
# a compact PixelCNN++ implementation that lives next to this notebook
from pixelcnn_pp_pytorch import PixelCNN

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
N_COMPONENTS = 5
EPOCHS = 10
BATCH_SIZE = 128

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data - Keras downloads it once (cached in ~/.keras/datasets)
(x_train, _), (_, _) = datasets.fashion_mnist.load_data()  # uint8 array of shape (60000, 28, 28)

In [ ]:
# Preprocess the data


def preprocess(imgs):
    # (N, 28, 28) -> (N, 1, 28, 28): PyTorch images are channels-first (NCHW)
    imgs = torch.tensor(imgs, dtype=torch.float32).unsqueeze(1)
    imgs = F.interpolate(
        imgs, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )
    # Keep pixel values in [0, 255], rounded to whole numbers because the
    # PixelCNN models discrete pixel values
    return imgs.round()


input_data = preprocess(x_train)

In [ ]:
# Show some items of clothing from the training set
display(input_data)

## 2. Build the PixelCNN <a name="build"></a>

In [ ]:
# Define a Pixel CNN network (see pixelcnn_pp_pytorch.py). Like the TFP version,
# it is a distribution over images with log_prob() and sample() methods
dist = PixelCNN(
    image_shape=(1, IMAGE_SIZE, IMAGE_SIZE),  # channels first
    num_resnet=1,
    num_hierarchies=2,
    num_filters=32,
    num_logistic_mix=N_COMPONENTS,
    dropout_p=0.3,
)

print(dist)
print(f"Total params: {sum(p.numel() for p in dist.parameters()):,}")


# The LightningModule wraps the distribution; the loss is the
# negative log-likelihood of the images
class LitPixelCNN(L.LightningModule):
    def __init__(self, dist, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters(ignore=["dist"])
        self.dist = dist

    def training_step(self, images, batch_idx):
        loss = -self.dist.log_prob(images).mean()
        self.log("loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss  # Lightning calls zero_grad / backward / step for us

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

## 3. Train the PixelCNN <a name="train"></a>

In [ ]:
# "Compiling" in Lightning = creating the LightningModule; its
# configure_optimizers() builds the Adam optimizer when training starts
pixelcnn = LitPixelCNN(dist, learning_rate=0.001)

In [ ]:
tensorboard_logger = TensorBoardLogger(save_dir="./logs")


# Samples images; Lightning calls on_train_epoch_end after each epoch
class ImageGenerator(L.Callback):
    def __init__(self, num_img):
        self.num_img = num_img

    def generate(self):
        return dist.sample(self.num_img).cpu().numpy()

    def on_train_epoch_end(self, trainer, pl_module):
        generated_images = self.generate()
        display(
            generated_images,
            n=self.num_img,
            save_to="./output/generated_img_%03d.png" % (trainer.current_epoch),
        )


img_generator_callback = ImageGenerator(num_img=2)

In [ ]:
train_loader = DataLoader(input_data, batch_size=BATCH_SIZE, shuffle=True)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    logger=tensorboard_logger,
    callbacks=[
        img_generator_callback,
        # The plain tqdm progress bar: the Rich one (used when `rich` is installed)
        # clashes with printing/plotting from a callback inside Jupyter
        TQDMProgressBar(),
    ],
)
trainer.fit(pixelcnn, train_loader)

## 4. Generate images <a name="generate"></a>

In [ ]:
generated_images = img_generator_callback.generate()

In [ ]:
display(generated_images, n=img_generator_callback.num_img)